In [99]:
# import libraries and api key for FRED

import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from fredapi import Fred
import yfinance as yf

load_dotenv()
fred = Fred(api_key=os.environ["FRED_API_KEY"])

In [100]:
# get the time series data from FRED and Yahoo Finance

icsa = fred.get_series("ICSA", observation_start='1995-01-01')
dgs2 = fred.get_series("DGS2", observation_start='1995-01-01')
dgs10 = fred.get_series("DGS10", observation_start='1995-01-01')
cpi = fred.get_series('CPIAUCSL', observation_start='1995-01-01')
vix = yf.download('^VIX', start='1995-01-01')['Close'].squeeze()

[*********************100%***********************]  1 of 1 completed


In [101]:
print(icsa.head(), dgs2.head(), dgs10.head(), vix.head())

1995-01-07    338000.0
1995-01-14    347000.0
1995-01-21    325000.0
1995-01-28    324000.0
1995-02-04    324000.0
dtype: float64 1995-01-02     NaN
1995-01-03    7.73
1995-01-04    7.62
1995-01-05    7.66
1995-01-06    7.64
dtype: float64 1995-01-02     NaN
1995-01-03    7.88
1995-01-04    7.82
1995-01-05    7.88
1995-01-06    7.87
dtype: float64 Date
1995-01-03    14.25
1995-01-04    13.53
1995-01-05    13.50
1995-01-06    13.13
1995-01-09    13.33
Name: ^VIX, dtype: float64


In [102]:
# create a date range index from 1996-01-01 to today with business day frequency, calculate yield spread, yearly inflation, and reindex all series on new index

idx = pd.date_range(start='1996-01-01', end='today', freq='B')

vix = vix.reindex(idx, method='ffill')

dgs2.index = pd.to_datetime(dgs2.index)
dgs10.index = pd.to_datetime(dgs10.index)
spread = (dgs10 - dgs2).reindex(idx, method='ffill')

icsa.index = pd.to_datetime(icsa.index)
icsa = icsa.reindex(idx, method='ffill')

cpi.index = pd.to_datetime(cpi.index)
cpiYoY = cpi.pct_change(12) * 100
cpiYoY = cpiYoY.reindex(idx, method='ffill')

df = pd.DataFrame({
    'icsa': icsa,
    'spread': spread,
    'vix': vix,
    'cpiYoY': cpiYoY
})

print(df.head())
print(df.tail())

                icsa  spread    vix    cpiYoY
1996-01-01  359000.0     NaN  12.52  2.790698
1996-01-02  359000.0    0.42  12.19  2.790698
1996-01-03  359000.0    0.41  12.10  2.790698
1996-01-04  359000.0    0.48  13.78  2.790698
1996-01-05  359000.0    0.49  13.58  2.790698
                icsa  spread        vix    cpiYoY
2026-06-26  216000.0    0.31  18.410000  4.166615
2026-06-29  215000.0    0.28  17.650000  4.166615
2026-06-30  215000.0    0.30  16.450001  4.166615
2026-07-01  215000.0    0.30  16.590000  4.166615
2026-07-02  215000.0    0.30  16.590000  4.166615


In [103]:
df_z = pd.DataFrame(index=df.index)

windows = {
    'icsa': 252,
    'spread': 252,
    'vix': 252,
    'cpiYoY': 1260
}

for col, window in windows.items():
    min_periods = int(window * 0.9)
    rollingMean = df[col].rolling(window, min_periods=min_periods).mean()
    rollingStd = df[col].rolling(window, min_periods=min_periods).std()
    df_z[col] = (df[col] - rollingMean) / rollingStd


df_z = df_z.dropna()

print(df_z)

                icsa    spread       vix    cpiYoY
2000-05-04  0.312858 -1.692789  2.825877  1.304183
2000-05-05  0.316941 -1.756334  1.541560  1.302632
2000-05-08  0.487929 -1.615050  1.787052  1.301087
2000-05-09  0.491674 -1.719857  2.047216  1.299548
2000-05-10  0.493921 -1.861623  2.390375  1.298013
...              ...       ...       ...       ...
2026-06-26 -0.266457 -2.745116  0.087755 -0.158202
2026-06-29 -0.346967 -3.021542 -0.142337 -0.157835
2026-06-30 -0.342910 -2.743011 -0.502332 -0.157467
2026-07-01 -0.340832 -2.691220 -0.459490 -0.157029
2026-07-02 -0.338757 -2.640777 -0.458403 -0.156591

[6521 rows x 4 columns]
